In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

from torch.nn.utils.fusion import fuse_conv_bn_eval

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import onnx

import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime
from time import time
import copy

In [2]:
import json_to_pytorch

In [3]:
def get_loaders(batch_size=64, num_workers=2, normalize=True):
    # Statistiche CIFAR-10 standard
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    if normalize:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
    else:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
      ])

    train_ds = datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
    test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

In [4]:
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total

In [5]:
model = json_to_pytorch.json_to_pytorch("../VeryDiffPolyExperiments/results/gelu/best_model_bn_8_0.0001l1_no_pad_50.json", double_precision=True)

In [6]:
model

Sequential(
  (0): FrozenBatchNorm()
  (1): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
  (2): ChebyshevPoly()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
  (4): ChebyshevPoly()
  (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2))
  (6): ChebyshevPoly()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): Linear(in_features=288, out_features=256, bias=True)
  (9): ChebyshevPoly()
  (10): Linear(in_features=256, out_features=10, bias=True)
)

In [7]:
train_loader, test_loader = get_loaders(batch_size=128)
test_loss,  test_acc  = evaluate(model, test_loader, criterion, "cpu")

Files already downloaded and verified
Files already downloaded and verified


In [8]:
test_acc

0.1004

In [10]:
# Get sample from test_loader
test_input, test_label = next(iter(test_loader))

In [17]:
test_input = torch.rand(test_input[0:1,:,:,:].shape)

In [18]:
torch.set_printoptions(threshold=30_000)

In [20]:
test_input

tensor([[[[5.3083e-01, 8.1572e-01, 6.5587e-01, 9.5627e-01, 7.3559e-02,
           6.0096e-01, 9.3190e-01, 4.0720e-01, 3.7502e-02, 2.7355e-01,
           8.3820e-01, 2.3372e-01, 9.5467e-02, 6.6738e-01, 8.1357e-01,
           7.5083e-01, 7.2920e-01, 8.4346e-01, 7.3757e-02, 8.1139e-01,
           7.9476e-01, 6.0158e-01, 7.7013e-01, 3.0864e-01, 9.7338e-01,
           2.5748e-01, 5.8568e-01, 3.7602e-01, 4.5951e-01, 5.9474e-01,
           3.8353e-01, 5.0550e-01],
          [5.7273e-01, 6.6153e-01, 9.0414e-01, 1.6038e-01, 3.2400e-01,
           3.0815e-01, 2.0041e-01, 9.7017e-01, 9.7660e-01, 5.2659e-01,
           9.2216e-01, 9.4682e-01, 1.4427e-01, 8.8793e-01, 7.5621e-01,
           1.3228e-01, 7.6920e-02, 2.0179e-01, 9.1863e-02, 2.3608e-01,
           9.8976e-01, 7.3318e-01, 9.2319e-01, 4.4494e-01, 9.8091e-01,
           4.6451e-01, 9.2024e-01, 7.8028e-01, 5.4447e-01, 1.5224e-01,
           3.7357e-01, 6.2692e-01],
          [4.6506e-01, 9.4095e-02, 2.2437e-01, 4.6818e-01, 2.9208e-01,
     

In [21]:
module_list = list(model.children())
module_list

[FrozenBatchNorm(),
 Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2)),
 ChebyshevPoly(),
 Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2)),
 ChebyshevPoly(),
 Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2)),
 ChebyshevPoly(),
 Flatten(start_dim=1, end_dim=-1),
 Linear(in_features=288, out_features=256, bias=True),
 ChebyshevPoly(),
 Linear(in_features=256, out_features=10, bias=True)]

In [22]:
model_part = nn.Sequential(*module_list[0:5])

In [23]:
model_part

Sequential(
  (0): FrozenBatchNorm()
  (1): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
  (2): ChebyshevPoly()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
  (4): ChebyshevPoly()
)

In [24]:
model(test_input[0:1])

tensor([[ 0.6755,  7.4113,  4.0774, -3.7454, -3.8706, -5.3125,  0.5390, -1.0061,
         -0.8500,  2.5709]], dtype=torch.float64, grad_fn=<AddmmBackward0>)